In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

forecasts = pd.read_csv("../results/forecasts.csv")
data = pd.read_csv("../data/raw/data.csv")

# Baseline only
forecasts = forecasts[
    forecasts["model"] == "baseline"
].copy()

forecasts["period"] = pd.to_datetime(forecasts["period"])

# Convert raw YYYYMM dates
data["Period"] = pd.to_datetime(
    data["Period"].astype(str),
    format="%Y%m"
)

# Match the economic condition to the month being predicted
df = forecasts.merge(
    data[["Country", "Period", "NL-CONJCLK"]],
    left_on=["country", "period"],
    right_on=["Country", "Period"],
    how="left",
    validate="many_to_one"
)

assert df["NL-CONJCLK"].notna().all()

df["absolute_error"] = abs(
    df["actual"] - df["prediction"]
)

display(df.head(15))

In [ ]:

df["business_cycle"] = np.where(
    df["NL-CONJCLK"] >= 0,
    "Favorable",
    "Unfavorable"
)

print(
    df.groupby("business_cycle")["origin"]
      .nunique()
)


In [ ]:

results = (
    df.groupby(
        ["country", "model", "horizon", "business_cycle"]
    )
    .agg(
        mae=("absolute_error", "mean"),
        total_error=("absolute_error", "sum"),
        total_actual=("actual", "sum"),
        n_forecasts=("actual", "size"),
    )
    .reset_index()
)

results["wape"] = (
    100 * results["total_error"] / results["total_actual"]
)

results = results.drop(
    columns=["total_error", "total_actual"]
)

print(results.round(2).to_string(index=False))

results.to_csv(
    "../results/business_cycle_metrics.csv",
    index=False
)


In [ ]:
for country in results["country"].unique():

    subset = results[
        results["country"] == country
    ]

    pivot = subset.pivot(
        index="horizon",
        columns="business_cycle",
        values="mae"
    )

    ax = pivot.plot(
        kind="line",
        marker="o",
        figsize=(7, 4)
    )

    ax.set_title(f"{country} — Baseline MAE")
    ax.set_xlabel("Forecasting Horizon (months)")
    ax.set_ylabel("Mean Absolute Error")
    ax.set_xticks([1, 3, 6])
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# One-month forecasts only
plot_df = df[df["horizon"] == 1].copy()

countries = ["Bulgaria", "Germany", "Italy", "Poland", "Romania", "Spain"]

fig, axes = plt.subplots(
    nrows=6,
    ncols=1,
    figsize=(12, 20),
    sharex=True
)

for ax, country in zip(axes, countries):

    subset = (
        plot_df[plot_df["country"] == country]
        .sort_values("period")
    )

    # Business-cycle background shading
    for _, row in subset.iterrows():

        color = (
            "green"
            if row["business_cycle"] == "Favorable"
            else "red"
        )

        ax.axvspan(
            row["period"],
            row["period"] + pd.DateOffset(months=1),
            color=color,
            alpha=0.12,
            linewidth=0
        )

    # Actual immigration
    ax.plot(
        subset["period"],
        subset["actual"],
        color="black",
        linewidth=2,
        label="Actual immigration"
    )

    # Predicted immigration
    ax.plot(
        subset["period"],
        subset["prediction"],
        color="blue",
        linestyle="--",
        linewidth=1.5,
        label="Baseline forecast"
    )

    ax.set_title(country)
    ax.set_ylabel("Monthly immigration")
    ax.grid(alpha=0.2)

# Shared legend for the entire figure
legend_items = [
    plt.Line2D([0], [0], color="black", linewidth=2,
               label="Actual immigration"),
    plt.Line2D([0], [0], color="blue", linestyle="--",
               label="Baseline forecast"),
    Patch(color="green", alpha=0.2, label="Favorable"),
    Patch(color="red", alpha=0.2, label="Unfavorable")
]

fig.legend(
    handles=legend_items,
    loc="upper center",
    ncol=4,
    bbox_to_anchor=(0.5, 0.965)
)

fig.suptitle(
    "Actual vs. Predicted Immigration by Business-Cycle Condition",
    fontsize=14,
    y=1.01
)

axes[-1].set_xlabel("Month")

plt.tight_layout(rect=[0, 0, 1, 0.93])

# Save for the Technical Report
plt.savefig(
    "../results/business_cycle_forecasts.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()